
# TRAINING SCRIPT

In [ ]:
import numpy as np
import pandas as pd

# LOAD DATA 

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/aryansharmabf/aqi-dataset1/delhi_air_quality_feature_store_unprocessed.csv")

In [ ]:
OUT_FILE ="delhi_2000_2025_extended.csv"

# reanme the columns 

In [ ]:
df.rename(columns={'event_timestamp': 'Time'}, inplace=True)
df.rename(columns={'city	':'city'},inplace = True)

In [ ]:
df.head(5)

In [ ]:
df['Time'] = pd.to_datetime(df['Time'],errors = "coerce")
df = df.dropna(subset = ['Time']).sort_values('Time').reset_index(drop = True)

In [ ]:
df.head(2)

# Ensure numeric

In [ ]:
num_cols = [
    "temperature","humidity","pressure","wind_speed","wind_direction",
    "pm25","pm10","no2","so2","o3","co","aqi"]
for c in num_cols:
    if c in df.columns :
        df[c] = pd.to_numeric(df[c],errors = "coerce")

# Take 2024 slice

In [ ]:
df_2024 =df[df['Time'].dt.year==2024].copy()
if len(df_2024) == 0:
     raise ValueError("No 2024 data found. Can't generate 2025 extension.")
df_2025 = df_2024.copy()
df_2025['Time'] =df_2025['Time']+pd.DateOffset(years = 1) # shift to 2025

# Controlled noise (keep structure; avoid crazy values)

In [ ]:
cols_to_noise = [
    "temperature","humidity","pressure","wind_speed","wind_direction",
    "pm25","pm10","no2","so2","o3","co","aqi"
]
for col in cols_to_noise:
    if col not in df_2025.columns:
        continue
        
    std = df_2025[col].std(skipna=True)
    if pd.isna(std) or std == 0:
        continue

    # Extract month
    months = df_2025["Time"].dt.month

    # Seasonal noise factor
    season_factor = np.where(
        months.isin([4,5,6]), 0.02,   # Summer → low variation
        np.where(months.isin([10,11,12]), 0.05, 0.03)  # Winter → higher variation
    )

    noise = np.random.normal(0, std * season_factor, len(df_2025))

    df_2025[col] = df_2025[col] + noise
    df_2025[col] = df_2025[col].clip(lower=0)
    
    

# Combine

In [ ]:
df_ext = pd.concat([df, df_2025], ignore_index=True)
df_ext = df_ext.sort_values("Time").reset_index(drop=True)

# out 2000 to 2025 Dataset

In [ ]:
df_ext.to_csv(OUT_FILE, index=False)
print(f"✅ Saved extended dataset: {OUT_FILE}")